# 01. 알약 데이터 EDA (통합본)

팀원별 EDA 노트북 6개를 주제별로 합친 노트북입니다. 같은 주제는 한 셀을 기준본으로 두고
나머지 노트북의 장점을 기준본에 녹였습니다. 섹션마다 첫 셀에 원작자와 원본 노트북을 적었습니다.

## 실행 방법

- `uv sync` 후 VS Code나 `uv run jupyter lab`에서 프로젝트 `.venv` 커널로 엽니다.
- 위에서 아래로 한 번 실행(Restart & Run All)합니다. 작업 폴더는 프로젝트 루트와 `notebooks/` 모두 됩니다.
- 원본은 `data/raw/`에서 읽기만 합니다. `ann_df`·`img_df`·`cat_df`는 원본 상태로 유지하고,
  보정 비교는 별도 `ann_corrected_df`에서 합니다.
- 커밋 전에 출력을 지웁니다(Clear All Outputs).

## 실행 옵션 (1장 설정 셀)

| 옵션 | 기본값 | 설명 |
|---|---|---|
| `RUN_HEAVY` | `False` | 전체 이미지 SHA256·디코딩, 전체 박스 GrabCut 같은 무거운 연산을 실행합니다. `False`이면 입력·버전·파라미터가 일치하는 캐시만 재사용하고, 없으면 `미실행`으로 표시합니다. `True`이면 캐시와 관계없이 다시 계산합니다. |
| `SAVE_OUTPUTS` | `False` | 그림·보고서·캐시 파일을 저장합니다. `False`이면 파일을 쓰지 않습니다. 켜도 원본은 수정하지 않습니다. |
| `SEED` | `42` | 무작위 연산 전체에 쓰는 seed입니다. |

`미실행` 결과는 0건이나 검사 통과로 해석하지 않습니다. 후속 셀은 결과 상태를 확인한 뒤 집계합니다.

## 데이터 상태 표시

표와 그림 제목 앞에 데이터 상태를 붙입니다. `[원본]`은 `data/raw/`를 그대로 읽은 결과,
`[보정]`은 보정 기록을 메모리에서 적용한 결과입니다.

## 섹션별 실행 시간

| 섹션 | `RUN_HEAVY=False` | `RUN_HEAVY=True` |
|---|---|---|
| 1–2. 설정·데이터 로드 | 약 3초 (Apple Silicon Mac) | 같음 |
| 3–4. 규모·구조·결측 | 약 2초 | 약 15초 (전체 디코딩·SHA256 13초) |
| 5–9 | 이식 후 측정 | 이식 후 측정 |

## 목차

0. 개요
1. 환경 설정
2. 데이터 로드
3. 데이터 규모와 구조
4. 결측
5. 라벨 오류와 이상치
6. 분포
7. 촬영 조건
8. 데이터 누수와 분할
9. 결론과 다음 작업

## 원본 노트북

| 노트북 | 작성자 |
|---|---|
| `EDA_jang.ipynb` | younghoon1215 |
| `JJAM99_eda.ipynb` | potato |
| `dongwook_eda.ipynb` | def-uk |
| `eda_kwon.ipynb` | gwon753-dev |
| `eda_sohee.ipynb`, `sohee_eda.ipynb` | bravedolphinhere |

## 1. 환경 설정

> 출처: dongwook_eda.ipynb 셀 1 (def-uk), eda_kwon.ipynb 셀 2, 35 (gwon753-dev),
> sohee_eda.ipynb 셀 1, 3 (bravedolphinhere)

- 경로는 모두 `src/config.py`의 `PROJECT_ROOT` 기준입니다.
- 한글 폰트는 Malgun Gothic → AppleGothic → NanumGothic → Noto Sans CJK KR 순서로 찾습니다.
  WSL에서는 Windows의 맑은 고딕 파일을 등록합니다.
- 무작위 연산은 모두 `SEED`를 씁니다(`random_state=SEED`, `np.random.default_rng(SEED)`).

In [ ]:
import sys
from pathlib import Path

# 노트북은 notebooks/에서 실행되므로 프로젝트 루트를 import 경로에 추가한다.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import hashlib
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from data_process.data_processing import (
    build_cat_df,
    build_img_df,
    fingerprint_files,
    inspect_image_integrity,
    list_images,
    load_annotations,
    load_cache,
    load_schema,
    read_image_headers,
    save_cache,
    scan_missing_values,
)
from src.config import PROJECT_ROOT
from visualization import draw_bboxes, image_grid, set_korean_font, show_image

SEED = 42
RUN_HEAVY = False  # True면 무거운 연산을 캐시와 관계없이 다시 계산한다.
SAVE_OUTPUTS = False  # True일 때만 그림·보고서·캐시 파일을 쓴다.
NOT_RUN = "미실행"  # 생략한 분석의 결과 상태. 0건이나 통과로 해석하지 않는다.

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"  # 받은 원본 그대로 (읽기만 함)
# 압축을 푼 위치에 따라 sprint_ai_project1_data 폴더가 한 단계 더 있을 수 있다.
if (RAW_DIR / "sprint_ai_project1_data").is_dir():
    RAW_DIR = RAW_DIR / "sprint_ai_project1_data"
PROCESSED_DIR = DATA_DIR / "processed"  # 보정본·보정 기록
CACHE_DIR = PROCESSED_DIR / "eda_cache"  # 무거운 연산 캐시. SAVE_OUTPUTS=True일 때만 쓴다.

TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"
for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR]:
    if not folder.is_dir():
        raise FileNotFoundError(f"데이터 폴더가 없습니다: {folder.relative_to(PROJECT_ROOT)}")

np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
font = set_korean_font()
if font is None:
    print("한글 폰트를 찾지 못했습니다. 약 이름이 네모로 보일 수 있습니다.")

print(f"SEED={SEED}, RUN_HEAVY={RUN_HEAVY}, SAVE_OUTPUTS={SAVE_OUTPUTS}, 폰트={font}")
print(f"원본 데이터: {RAW_DIR.relative_to(PROJECT_ROOT).as_posix()}/")

## 2. 데이터 로드

> 출처: dongwook_eda.ipynb 셀 0–3 (def-uk) 기준. eda_kwon.ipynb 셀 1–2 (gwon753-dev)의
> 디스크 기준 `img_df`·`cat_df`·`test_files`, sohee_eda.ipynb 셀 2, 7 (bravedolphinhere)의
> JSON 내부 참조 연결과 읽기 오류 기록을 합쳤습니다. 로더는 `data_process/data_processing.py`에 있습니다.

- annotation JSON 하나 = 이미지 1장 × 알약 1개입니다. 같은 이미지의 알약이 약 코드별 폴더에 흩어져 있습니다.
  경로: `train_annotations/<조합>_json/<약코드>/<이미지이름>.json`
- bbox는 COCO 형식 `[x, y, w, h]`(왼쪽 위 기준, 픽셀)입니다.
- 이미지 메타데이터는 JSON 안의 `annotation.image_id`로 연결합니다. 전역 `image_id`로 합치지 않습니다.
- 라벨 식별자는 `(source_json, annotation_id)`입니다. `source_json`은 `train_annotations/` 기준 상대 경로입니다.
- bbox 값이 비어 있는 라벨도 빼지 않고 `x, y, w, h`를 NaN으로 둡니다(4.1에서 확인).
- 읽기·연결 실패는 `load_error_df`에 남기고, 분석에서 제외된 입력 수를 보여 줍니다.

| 표 | 한 행 | 설명 |
|---|---|---|
| `ann_df` | 원본 bbox 1개 | 이미지 메타데이터 + 약 정보 + bbox. 원본 상태를 유지합니다. |
| `img_df` | 디스크의 train 이미지 1장 | 라벨 없는 이미지도 포함합니다(`n_boxes=0`). `combo`는 파일 이름의 알약 조합입니다. |
| `cat_df` | 클래스 1개 | `category_id` ↔ 약 이름, bbox 수, 이미지 수 |
| `test_files` | test 이미지 1장 | 파일 이름 목록 |
| `load_error_df` | 오류 1건 | `stage`: `parse`(JSON 읽기), `image_link`, `category_link`, `empty`(annotation 없음) |

In [ ]:
ann_df, load_error_df = load_annotations(TRAIN_ANN_DIR)
img_df = build_img_df(ann_df, list_images(TRAIN_IMG_DIR))
cat_df = build_cat_df(ann_df)
test_files = [p.name for p in list_images(TEST_IMG_DIR)]

# 제외된 입력: parse 오류는 JSON 파일 전체, image_link 오류는 annotation 하나가 빠진다.
n_json = sum(1 for _ in TRAIN_ANN_DIR.rglob("*.json"))
excluded = load_error_df[load_error_df["excluded"]]
n_json_excluded = (excluded["stage"] == "parse").sum()
n_ann_excluded = (excluded["stage"] == "image_link").sum()
n_no_image = (~ann_df["file_name"].isin(img_df["file_name"])).sum()

print(f"[원본] annotation JSON: {n_json}개 (읽기 실패로 제외: {n_json_excluded}개)")
print(
    f"[원본] bbox: {len(ann_df)}개 (연결 실패로 제외: {n_ann_excluded}개), 클래스: {len(cat_df)}개"
)
print(
    f"[원본] train 이미지: {len(img_df)}장 (bbox 없음: {(img_df['n_boxes'] == 0).sum()}장), "
    f"test 이미지: {len(test_files)}장"
)
print(f"[원본] 디스크에 이미지 파일이 없는 bbox: {n_no_image}개")
print(f"로드 오류: {len(load_error_df)}건")
if not load_error_df.empty:
    display(load_error_df)

In [ ]:
display(ann_df.head(3))
display(img_df.head(3))
display(cat_df.head(3))

### 2.1 데이터 정의서

> 출처: eda_kwon.ipynb 셀 3 (gwon753-dev) 기준. dongwook_eda.ipynb 셀 9 (def-uk)의
> 스키마 필수여부 읽기를 합쳤습니다.

먼저 `data/README.md`의 스키마 표와 실제 JSON 키를 대조하고, `ann_df` 컬럼마다 필수여부(M 필수,
O 선택, C 조건부)와 스키마 설명을 붙입니다. 결측 비율은 4.3에서 봅니다.

In [ ]:
schema_df = load_schema(DATA_DIR / "README.md")
# 최상위 키(images, annotations 등)는 field가 비어 있어 section만 키로 쓴다.
schema_df["key"] = (schema_df["section"] + "." + schema_df["field"]).str.rstrip(".")

# ann_df 컬럼 → 스키마 키. 로더가 바꾸거나 나눈 컬럼만 적고, 나머지는 images[]의 같은 이름.
LOADER_COLS = {
    "image_id": "images.id",
    "source_json": None,  # 로더가 붙인 원본 JSON 상대 경로
    "annotation_id": "annotations.id",
    "category_id": "annotations.category_id",
    "category_name": "categories.name",
    "x": "annotations.bbox",
    "y": "annotations.bbox",
    "w": "annotations.bbox",
    "h": "annotations.bbox",
    "area": "annotations.area",
    "iscrowd": "annotations.iscrowd",
    "ignore": "annotations.ignore",
    "has_segmentation": "annotations.segmentation",
}
col_to_key = {c: LOADER_COLS.get(c, f"images.{c}") for c in ann_df.columns}

json_image_keys = {k for c, k in col_to_key.items() if k and k.startswith("images.")}
is_image_field = (schema_df["section"] == "images") & (schema_df["field"] != "")
schema_image_keys = set(schema_df.loc[is_image_field, "key"])
print("스키마에 없는 JSON 키:", sorted(json_image_keys - schema_image_keys) or "없음")
print("JSON에 없는 스키마 키:", sorted(schema_image_keys - json_image_keys) or "없음")

definition_df = pd.DataFrame(
    {
        "컬럼": ann_df.columns,
        "스키마 키": [col_to_key[c] for c in ann_df.columns],
        "dtype": ann_df.dtypes.astype(str).to_numpy(),
        "고유값 수": ann_df.nunique().to_numpy(),
        "예시": [ann_df[c].dropna().iloc[0] if ann_df[c].notna().any() else None for c in ann_df],
    }
)
definition_df = definition_df.merge(
    schema_df[["key", "type", "required", "description"]],
    left_on="스키마 키",
    right_on="key",
    how="left",
).drop(columns="key")
definition_df = definition_df.rename(
    columns={"type": "스키마 타입", "required": "필수여부", "description": "스키마 설명"}
)
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 40):
    display(definition_df)

아래는 kwon이 값을 보고 정리한 컬럼 해석입니다. 수치는 원본 상태 관측값이며 3–4장에서 다시 확인합니다.
필수여부는 위 표를 따릅니다.

### 이미지 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| file_name | str | 이미지 파일명. 이미지 단위의 키로 사용됩니다. 앞부분(`K-003351-018357-036637`)에 사진에 찍힌 약 코드가 들어 있습니다. |
| imgfile | str | 이미지 파일명. `file_name`과 같은 값입니다. 스키마 표에는 없는 키입니다. |
| width | int | 이미지 너비(px). 전부 976입니다. |
| height | int | 이미지 높이(px). 전부 1280입니다. |
| image_id | int | JSON 안의 이미지 아이디(`images[].id`)입니다. |
| source_json | str | 이 행을 읽어 온 annotation JSON의 상대 경로입니다(`train_annotations/` 기준). |

### 촬영 조건

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| back_color | str | 배경색. 전부 `연회색 배경`입니다. |
| light_color | str | 조명 색. 전부 `주백색`입니다. |
| camera_la | int | 카메라 촬영 각도(위도). 70, 75, 90 세 가지입니다. |
| camera_lo | int | 카메라 회전 각도(경도). 전부 0입니다. |
| size | int | 촬영 크기 설정값. 전부 200입니다. |
| drug_dir | str | 촬영된 알약 면. `앞면` 또는 `뒷면`입니다. |
| drug_S | str | 알약 상태. 전부 `정상알약`입니다. |

### 정답 (타깃)

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| annotation_id | int | 어노테이션 아이디. `source_json`과 함께 라벨 하나를 가리킵니다. |
| category_id | int | **타깃 변수(클래스)**. 약 종류 코드이며 56종입니다. `drug_N`의 숫자 부분과 같습니다. |
| category_name | str | JSON `categories[].name`의 약 이름입니다. |
| x | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 x 좌표(px)입니다. |
| y | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 y 좌표(px)입니다. |
| w | int | **타깃 변수(bbox)**. 박스 너비(px)입니다. |
| h | int | **타깃 변수(bbox)**. 박스 높이(px)입니다. |
| area | int | 박스 넓이(px²). 대부분 `w × h`와 같습니다. 다른 경우는 5.4에서 봅니다. |
| iscrowd, ignore, has_segmentation | int, int, bool | 평가 분류, 무시 여부, segmentation 유무입니다. |

### 약 식별 코드

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| drug_N | str | 약 코드(예: `K-003351`). `category_id`에 `K-`를 붙인 형태입니다. |
| dl_mapping_code | str | 약 매핑 코드. `drug_N`과 같은 값입니다. |
| dl_idx | str | 약 코드의 숫자 부분(예: `3351`)입니다. (이 부분은 일단 눈여겨 보자) |
| item_seq | int | 식약처 품목기준코드입니다. |
| di_edi_code | str | 건강보험 EDI 코드. 여러 개면 쉼표로 구분합니다. |

### 약 기본 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_name | str | 약 이름(한글)입니다. |
| dl_name_en | str | 약 이름(영문). 14행이 빈 문자열입니다. |
| dl_material | str | 주성분(한글). 성분이 여러 개면 `\|`로 구분합니다. 9행이 빈 문자열입니다. |
| dl_material_en | str | 주성분(영문)입니다. |
| dl_company | str | 제약회사(한글)입니다. |
| dl_company_en | str | 제약회사(영문)입니다. |
| di_company_mf | str | 위탁 제조사(한글). 대부분(637행) 빈 문자열입니다. |
| di_company_mf_en | str | 위탁 제조사(영문). 대부분(724행) 빈 문자열입니다. |
| di_class_no | str | 약효 분류(예: `[02180]동맥경화용제`)입니다. |
| di_etc_otc_code | str | 의약품 구분. `전문의약품` 또는 `일반의약품`입니다. |
| di_item_permit_date | str | 품목 허가일(YYYYMMDD)입니다. |

### 약 외형

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_custom_shape | str | 제형 분류(예: `정제, 저작정`, `경질캡슐제`)입니다. |
| form_code_name | str | 제형 이름(예: `필름코팅정`, `나정`)입니다. |
| drug_shape | str | 알약 모양(예: `원형`, `타원형`, `장방형`)입니다. |
| chart | str | 성상. 알약 생김새를 글로 설명한 것입니다. |
| color_class1 | str | 주 색상입니다. |
| color_class2 | str | 보조 색상. 대부분 빈 문자열입니다. |
| thick | float | 두께(mm). 0.0은 값이 없는 경우입니다. |
| leng_long | float | 장축 길이(mm). 0.0은 값이 없는 경우입니다. |
| leng_short | float | 단축 길이(mm). 0.0은 값이 없는 경우입니다. |
| print_front | str | 앞면 각인 문자(예: `R5`)입니다. |
| print_back | str | 뒷면 각인 문자. `분할선`처럼 적힌 경우도 있습니다. |
| line_front | str | 앞면 분할선. `-`(일자), `+`(십자), 빈 문자열(없음)입니다. |
| line_back | str | 뒷면 분할선. 값은 `line_front`와 같은 방식입니다. |

### 마크 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| mark_code_front_anal | str | 앞면 마크 설명(예: `pfizer`). 대부분 빈 문자열입니다. |
| mark_code_front_img | str | 앞면 마크 이미지 URL. 대부분 빈 문자열입니다. |
| mark_code_back_anal | str | 뒷면 마크 설명. 전부 비어 있습니다. |
| mark_code_back_img | str | 뒷면 마크 이미지 URL. 전부 비어 있습니다. |
| mark_code_front | str | 앞면 마크 코드. 전부 비어 있습니다. |
| mark_code_back | str | 뒷면 마크 코드. 전부 비어 있습니다. |

### 기타

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| img_key | str | 공공 낱알 이미지 URL입니다. |
| img_regist_ts | str | 낱알 이미지 등록일(YYYYMMDD)입니다. |
| change_date | str | 약 정보 변경일(YYYYMMDD)입니다. |

> 참고: `img_key`, `chart`, `drug_shape`, `print_*`, `color_*`, `line_*`, `mark_*`, `img_regist_ts`,
> `change_date`, `form_code_name`은 18행이 결측(NaN)입니다. 키는 있고 값이 `null`입니다(4.3에서 확인).
> 원본 kwon 정의서와 dongwook 셀 9는 이를 "키 없음"으로 적었습니다.

## 3. 데이터 규모와 구조

> 출처: JJAM99_eda.ipynb 셀 3–17, 19 (potato) 기준. sohee_eda.ipynb 셀 7, 9, 11 (bravedolphinhere)의
> image_id 대응 확인, 조합별 예시 이미지, 디코딩 실패·SHA256·train/test 공통 해시를 합쳤습니다.

### 3.1 파일 수, 해상도, 색상 모드, 디코딩 오류

> 출처: JJAM99_eda.ipynb 셀 3, 5, 19 (potato) 기준. sohee_eda.ipynb 셀 11 (bravedolphinhere)의
> 디코딩 실패·SHA256을 합쳤습니다.

먼저 이미지 헤더만 읽어 해상도와 색상 모드를 봅니다(빠름). 헤더가 정상이어도 픽셀 손상까지 확인된
것은 아니므로, 전체 디코딩과 SHA256은 다음 셀에서 `RUN_HEAVY`일 때만 계산합니다.
JSON에 적힌 `width`·`height`와 실제 이미지 크기도 라벨마다 대조합니다.

In [ ]:
img_paths = {"train": list_images(TRAIN_IMG_DIR), "test": list_images(TEST_IMG_DIR)}
header_df = pd.concat(
    [read_image_headers(paths, split) for split, paths in img_paths.items()], ignore_index=True
)
header_ok = header_df["error"].isna()
header_df["resolution"] = [
    f"{int(w)}×{int(h)}" if ok else None
    for w, h, ok in zip(header_df["width"], header_df["height"], header_ok, strict=True)
]
header_summary = header_df.groupby("split", sort=False).agg(
    n_images=("file_name", "size"),
    resolution=("resolution", lambda s: sorted(s.dropna().unique())),
    mode=("mode", lambda s: sorted(s.dropna().unique())),
    n_errors=("error", lambda s: s.notna().sum()),
)
print("[원본] 이미지 헤더")
display(
    header_summary.rename(
        columns={
            "n_images": "이미지 수",
            "resolution": "해상도",
            "mode": "색상 모드",
            "n_errors": "헤더 읽기 오류",
        }
    )
)
if not header_ok.all():
    display(header_df[~header_ok])

# 라벨마다 JSON의 width·height와 실제 이미지 크기를 비교한다. 헤더를 못 읽은 이미지는 위에서 셌다.
train_header = header_df[(header_df["split"] == "train") & header_ok]
size_check = ann_df[["source_json", "file_name", "width", "height"]].merge(
    train_header[["file_name", "width", "height"]], on="file_name", suffixes=("_json", "_actual")
)
size_mismatch = size_check[
    (size_check["width_json"] != size_check["width_actual"])
    | (size_check["height_json"] != size_check["height_actual"])
]
print(f"[원본] JSON 크기와 실제 이미지 크기가 다른 라벨: {len(size_mismatch)}개")
if not size_mismatch.empty:
    display(size_mismatch)

In [ ]:
# [RUN_HEAVY] 전체 디코딩·SHA256. 입력 지문·계산 버전이 저장 당시와 같은 캐시만 재사용한다.
INTEGRITY_VERSION = 1  # inspect_image_integrity의 계산 방식을 바꾸면 올린다.
integrity_cache = CACHE_DIR / "image_integrity.json"
integrity_key = {
    "version": INTEGRITY_VERSION,
    "data_state": "원본",
    "inputs": fingerprint_files([*img_paths["train"], *img_paths["test"]], RAW_DIR),
}
if RUN_HEAVY:
    start = time.perf_counter()
    integrity_df = pd.concat(
        [inspect_image_integrity(paths, split) for split, paths in img_paths.items()],
        ignore_index=True,
    )
    integrity_status = f"실행 ({time.perf_counter() - start:.0f}초)"
    if SAVE_OUTPUTS:
        save_cache(integrity_cache, integrity_key, integrity_df)
else:
    integrity_df, integrity_status = load_cache(integrity_cache, integrity_key)
    if integrity_df is None:
        integrity_status = f"{NOT_RUN} ({integrity_status})"

# integrity_df가 None이면 미실행이다. 뒤 셀은 None을 확인하고, 0건으로 바꿔 세지 않는다.
print(f"[원본] 전체 디코딩·SHA256: {integrity_status}")
if integrity_df is not None:
    decoded = integrity_df["decoded"].astype(bool)
    display(
        integrity_df.assign(failed=~decoded)
        .groupby("split", sort=False)
        .agg(n_images=("file_name", "size"), n_failed=("failed", "sum"))
        .rename(columns={"n_images": "이미지 수", "n_failed": "디코딩 실패"})
    )
    if not decoded.all():
        display(integrity_df[~decoded])

### 3.2 1 사진 : N JSON 구조와 bbox 시각 확인

> 출처: JJAM99_eda.ipynb 셀 7–14 (potato) 기준. sohee_eda.ipynb 셀 7, 9 (bravedolphinhere)의
> image_id 대응 확인과 여러 장 예시를 합쳤습니다.

- 사진과 JSON은 JSON 안의 `images[].file_name`으로 연결합니다. JSON 파일명이나 전역 `image_id`로
  합치지 않습니다. 그래도 둘이 사진과 어떻게 대응하는지는 확인해 둡니다.
- 학습할 때는 같은 사진을 가리키는 JSON의 라벨을 모두 모아야 합니다.

In [ ]:
# 사진당 JSON 수와 bbox 수. JSON 하나에 bbox가 하나씩이면 대각선에만 값이 있다.
print("[원본] 사진당 JSON 수 × bbox 수 (값: 사진 수)")
display(
    pd.crosstab(img_df["n_json"], img_df["n_boxes"]).rename_axis(index="JSON 수", columns="bbox 수")
)
boxes_per_json = ann_df.groupby("source_json").size()
n_missing_image = ann_df.loc[~ann_df["file_name"].isin(img_df["file_name"]), "file_name"].nunique()
print(f"[원본] bbox가 2개 이상인 JSON: {(boxes_per_json > 1).sum()}개")
print(f"[원본] JSON이 없는 train 이미지: {(img_df['n_json'] == 0).sum()}장")
print(f"[원본] JSON이 가리키지만 디스크에 없는 이미지: {n_missing_image}장")

# 연결 기준이 아닌 두 값도 사진과 1:1인지 본다. 1:1이어도 연결에는 file_name을 쓴다.
ids_per_file = ann_df.groupby("file_name")["image_id"].nunique()
files_per_id = ann_df.groupby("image_id")["file_name"].nunique()
print(
    f"[원본] image_id가 2개 이상인 사진: {(ids_per_file > 1).sum()}장 / "
    f"사진이 2장 이상인 image_id: {(files_per_id > 1).sum()}개"
)
json_stem = ann_df["source_json"].str.rsplit("/", n=1).str[-1].str.removesuffix(".json")
img_stem = ann_df["file_name"].str.rsplit(".", n=1).str[0]
print(f"[원본] JSON 파일명이 이미지 파일명과 다른 라벨: {(json_stem != img_stem).sum()}개")

In [ ]:
# bbox 수가 다른 사진을 한 장씩 골라 그 사진의 JSON 라벨을 모두 그린다.
sample_names = (
    img_df[img_df["n_boxes"] > 0]
    .drop_duplicates("n_boxes")
    .sort_values("n_boxes")["file_name"]
    .tolist()
)
fig, axes = image_grid(len(sample_names), cols=4)
for ax, name in zip(axes, sample_names, strict=True):
    boxes = ann_df[ann_df["file_name"] == name]
    boxes = boxes.assign(label=boxes["category_id"].astype(str) + " " + boxes["dl_name"])
    title = f"[원본] {name}\nJSON {boxes['source_json'].nunique()}개 / bbox {len(boxes)}개"
    show_image(ax, TRAIN_IMG_DIR / name, title, fontsize=7)
    draw_bboxes(ax, boxes, label_col="label")
plt.tight_layout()
plt.show()

- JSON 하나에는 사진 1장과 그 사진 속 알약 1개의 라벨이 들어 있습니다. 같은 사진의 라벨이
  약 코드별 폴더에 흩어져 있으므로 사진 단위로 모아야 합니다(JJAM 셀 8).
- JSON 파일명은 이미지 파일명과 같아서, 파일명이 같은 JSON이 사진 수만큼 묶입니다. 이것은 구조이지
  중복이 아닙니다(JJAM 셀 17). 내용 중복은 3.3에서 따로 봅니다.
- bbox가 2개인 사진 7장은 모두 파일명에 약이 3개인 사진입니다. 하나씩 라벨이 빠진 것으로,
  4.2에서 봅니다. bbox 3개 사진 중 1장도 파일명에는 약이 4개입니다(원본 상태 관측값).
- 그림의 박스는 JSON에 적힌 좌표입니다. 클래스 ID가 표시된다고 종류까지 맞다는 뜻은 아니며,
  그림만으로 라벨이 빠짐없이 있는지 확정하지 않습니다(라벨 누락은 4.2).

### 3.3 JSON 내용·라벨·픽셀 중복, train/test 중복

> 출처: JJAM99_eda.ipynb 셀 16, 19 (potato) 기준. sohee_eda.ipynb 셀 11 (bravedolphinhere)의
> train/test 공통 SHA256을 합쳤습니다.

파일명이 같다는 이유만으로 중복으로 보지 않습니다. 아래를 따로 셉니다.

| 검사 | 같다고 보는 기준 | 같으면 |
|---|---|---|
| JSON 내용 중복 | 키 순서·공백을 무시한 JSON 전체 | 같은 라벨 파일이 두 번 들어 있음 |
| 라벨 중복 | 사진·`category_id`·bbox | 같은 알약을 두 번 셈 |
| train/test 같은 파일명 | 파일명 | 같은 사진일 수 있음 |
| 파일·픽셀 중복 [RUN_HEAVY] | 파일 SHA256, RGB 픽셀 SHA256 | 같은 사진이 두 번 들어 있음. train/test 사이면 누수 |

회전·재촬영처럼 비슷하기만 한 사진은 8.2(유사도)에서, 다른 약이 같은 좌표에 붙은 경우는 5.2에서 봅니다.

In [ ]:
# JSON 내용 중복: 키 순서·공백을 무시하고 전체 내용이 같은 JSON을 묶는다.
json_hashes = {}
for json_path in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    try:
        data = json.loads(json_path.read_text(encoding="utf-8-sig"))
    except (OSError, UnicodeDecodeError, ValueError):
        continue  # 읽기 실패는 2장의 load_error_df에 있다.
    normalized = json.dumps(data, sort_keys=True, ensure_ascii=False)
    source_json = json_path.relative_to(TRAIN_ANN_DIR).as_posix()
    json_hashes[source_json] = hashlib.sha256(normalized.encode()).hexdigest()
json_hash = pd.Series(json_hashes, name="content_sha256")
dup_json = json_hash[json_hash.duplicated(keep=False)].sort_values()

# 라벨 중복: 같은 사진에 같은 클래스·같은 bbox가 두 번 이상 있는 경우
label_key = ["file_name", "category_id", "x", "y", "w", "h"]
dup_labels = ann_df[ann_df.duplicated(label_key, keep=False)].sort_values(label_key)
same_name = sorted(set(img_df["file_name"]) & set(test_files))


def count_hash_groups(df: pd.DataFrame, col: str) -> dict[str, int]:
    """해시가 같은 이미지 묶음 수를 train 안, test 안, train/test 사이로 나눠 센다."""
    splits = df.dropna(subset=[col]).groupby(col)["split"].agg(list)
    return {
        "train 안": int(splits.map(lambda s: s.count("train") > 1).sum()),
        "test 안": int(splits.map(lambda s: s.count("test") > 1).sum()),
        "train/test 사이": int(splits.map(lambda s: {"train", "test"} <= set(s)).sum()),
    }


dup_rows = [
    ("JSON 내용 중복 (묶음)", dup_json.nunique(), "완료"),
    ("라벨 중복: 사진·클래스·bbox (묶음)", dup_labels.groupby(label_key).ngroups, "완료"),
    ("train/test 같은 파일명 (장)", len(same_name), "완료"),
]
for col, label in [("file_sha256", "파일 SHA256"), ("pixel_sha256", "픽셀 SHA256")]:
    if integrity_df is None:
        groups = dict.fromkeys(["train 안", "test 안", "train/test 사이"], NOT_RUN)
    else:
        groups = count_hash_groups(integrity_df, col)
    for scope, value in groups.items():
        dup_rows.append((f"{label} 중복: {scope} (묶음)", value, integrity_status))
dup_check_df = pd.DataFrame(dup_rows, columns=["검사", "결과", "상태"])

print("[원본] 중복 검사")
display(dup_check_df)
for title, df in [("JSON 내용 중복", dup_json), ("라벨 중복", dup_labels)]:
    if not df.empty:
        print(title)
        display(df)

- JSON 내용 중복과 같은 사진·클래스·bbox의 라벨 중복은 없습니다(JJAM 셀 16, 19 관측과 같음).
- train/test 사이에 같은 파일명은 없습니다. 파일·픽셀이 완전히 같은 사진도 기존 관측에서는 없었습니다
  (JJAM 셀 19: 동일 픽셀 0묶음, sohee 결과: 학습/테스트 공통 SHA256 0). `RUN_HEAVY=False`이고 캐시가
  없으면 위 표에 `미실행`으로 나오며, 이 경우 중복이 없다고 확인된 것이 아닙니다.
- 완전 중복이 없다고 누수가 없는 것은 아닙니다. 같은 알약 조합을 각도만 바꿔 찍은 사진은 8장에서 봅니다.

## 4. 결측

> 출처: dongwook_eda.ipynb 셀 5–9 (def-uk), eda_kwon.ipynb 셀 4–8, 11–12 (gwon753-dev),
> JJAM99_eda.ipynb 셀 32–37 (potato), eda_sohee.ipynb 셀 6–14 (bravedolphinhere)

결측을 세 층으로 나눕니다.

| 절 | 무엇이 비었나 | 학습 영향 |
|---|---|---|
| 4.1 bbox 값 결측 | 라벨은 있는데 좌표·클래스 값이 없음 | 그 라벨을 쓸 수 없음 |
| 4.2 라벨 누락 | 사진에 있는 알약의 라벨 자체가 없음 | 빠진 알약이 배경으로 학습됨 |
| 4.3 메타데이터 결측 | 약 설명 키(각인, 색상, 마크 등)가 비어 있음 | bbox·클래스 학습에는 직접 영향 없음 |

### 4.1 bbox 값 결측

> 출처: dongwook_eda.ipynb 셀 6 (def-uk)의 값 결측 검사 기준. JJAM99_eda.ipynb 셀 33 (potato)의
> bbox 내부 결측 검사를 합쳤습니다.

로더는 bbox가 비었거나 숫자 4개가 아닌 라벨을 빼지 않고 `x, y, w, h`를 NaN으로 둡니다.
크기 0 이하, 이미지 밖 좌표, `area` 불일치처럼 값은 있지만 틀린 경우는 5장에서 봅니다.

In [ ]:
bbox_cols = ["x", "y", "w", "h", "area", "category_id", "category_name"]
print("[원본] bbox·클래스 컬럼별 결측 수")
display(ann_df[bbox_cols].isna().sum().rename("결측 수").to_frame().T)
n_category_link = (load_error_df["stage"] == "category_link").sum()
print(f"[원본] categories에서 category_id를 찾지 못한 라벨: {n_category_link}개")

bbox_missing = ann_df[ann_df[bbox_cols].isna().any(axis=1)]
if not bbox_missing.empty:
    display(bbox_missing[["source_json", "annotation_id", *bbox_cols]])

### 4.2 라벨 누락 (파일명에 있는 약의 bbox 없음)

> 출처: eda_kwon.ipynb 셀 4–8, 11–12 (gwon753-dev) 기준. dongwook_eda.ipynb 셀 6 (def-uk)의
> 같은 검사로 결과를 대조했습니다. kwon 셀 11의 약별 누락 비율 표는 손으로 쓴 표 대신 계산합니다.

파일명 앞부분(`K-003351-018357-036637`)에는 사진에 찍힌 약 코드가 들어 있습니다. 파일명에는 있는데
bbox가 없는 약을 라벨 누락으로 봅니다. 반대로 파일명에 없는 약의 bbox가 있는지도 확인합니다.

- 초록 박스: 라벨된 bbox (숫자는 `category_id`)
- 제목의 `누락`: bbox가 빠진 약 코드와 약 이름

In [ ]:
# 파일명의 약 코드(K-003351 → 3351)와 실제로 라벨된 category_id를 사진마다 비교한다.
name_codes = img_df["combo"].str.split("-").str[1:].map(lambda codes: {int(c) for c in codes})
labeled_by_file = ann_df.groupby("file_name")["category_id"].apply(set)
labeled_codes = img_df["file_name"].map(lambda name: labeled_by_file.get(name, set()))
label_check = img_df[["file_name", "combo", "n_codes_in_name", "n_boxes"]].assign(
    missing_codes=[sorted(n - lab) for n, lab in zip(name_codes, labeled_codes, strict=True)],
    extra_codes=[sorted(lab - n) for n, lab in zip(name_codes, labeled_codes, strict=True)],
)
missing_df = label_check[label_check["missing_codes"].str.len() > 0].reset_index(drop=True)
extra_df = label_check[label_check["extra_codes"].str.len() > 0].reset_index(drop=True)

print(
    f"[원본] 라벨이 빠진 이미지: {len(missing_df)}장 / {len(img_df)}장, "
    f"빠진 bbox: {missing_df['missing_codes'].str.len().sum()}개 "
    f"(그 사진들에 라벨된 bbox: {missing_df['n_boxes'].sum()}개)"
)
print(f"[원본] 파일명에 없는 약의 bbox가 있는 이미지: {len(extra_df)}장")
display(missing_df.drop(columns="extra_codes"))
if not extra_df.empty:
    display(extra_df.drop(columns="missing_codes"))

# 약별 누락 비율 = 빠진 bbox ÷ (라벨된 bbox + 빠진 bbox)
missing_by_drug = (
    missing_df["missing_codes"]
    .explode()
    .astype(int)
    .value_counts()
    .rename_axis("category_id")
    .rename("n_missing")
    .reset_index()
    .merge(cat_df[["category_id", "dl_name", "n_boxes"]], on="category_id", how="left")
)
missing_by_drug["누락 비율(%)"] = (
    100 * missing_by_drug["n_missing"] / (missing_by_drug["n_boxes"] + missing_by_drug["n_missing"])
).round(1)
print("[원본] 약별 누락")
display(missing_by_drug.rename(columns={"n_missing": "빠진 bbox", "n_boxes": "라벨된 bbox"}))

In [ ]:
# 라벨이 빠진 사진에 라벨된 bbox를 그리고, 제목에 빠진 약을 적는다.
drug_names = dict(zip(cat_df["category_id"], cat_df["dl_name"], strict=True))
if missing_df.empty:
    print("[원본] 라벨이 빠진 이미지가 없습니다.")
else:
    fig, axes = image_grid(len(missing_df), cols=4)
    for ax, row in zip(axes, missing_df.itertuples(), strict=True):
        missing = "\n".join(f"{c} {drug_names.get(c, '?')}" for c in row.missing_codes)
        show_image(ax, TRAIN_IMG_DIR / row.file_name, f"[원본] {row.combo}\n누락: {missing}")
        draw_bboxes(ax, ann_df[ann_df["file_name"] == row.file_name], color="green")
    plt.tight_layout()
    plt.show()

사진 전체의 라벨이 없는 것이 아니라, 사진 속 알약 하나의 bbox만 빠져 있습니다(kwon 셀 8).
그대로 학습하면 다음 문제가 생길 수 있습니다.

1. 빠진 알약을 배경으로 학습합니다. 모델이 알약을 덜 검출하는(미탐) 쪽으로 기울 수 있습니다.
2. 같은 약의 정답이 헷갈립니다. 예를 들어 3351은 다른 사진에서는 알약으로 라벨되어 있는데,
   누락된 사진에서는 배경으로 취급됩니다. 그 약에 대한 확신이 낮아질 수 있습니다.
3. 이 사진이 validation에 들어가면, 모델이 빠진 알약을 맞게 찾아도 오답(false positive)으로 채점됩니다.
4. 사진을 섞는 증강(mosaic 등)이 bbox 없는 알약 조각을 여러 학습 이미지로 퍼뜨립니다.

**처리 방안 (결정 전, kwon 셀 12)**

1. 그대로 쓰기
2. 빠진 bbox를 직접 그려 넣기 (kwon의 `added_annotations.csv`)
3. 해당 사진을 전부 빼기 (같은 사진의 멀쩡한 bbox도 함께 잃음)
4. 최소한 이 사진들은 validation에서 빼기

이 노트북은 누락을 찾고 보여 주기만 합니다. 보정 기록의 형식과 적용 방식은 5.8에서 정리합니다.

### 4.3 메타데이터 결측

> 출처: dongwook_eda.ipynb 셀 8–9 (def-uk) 기준. JJAM99_eda.ipynb 셀 33, 35, 37 (potato)의 결측 유형
> 구분·null 레코드 집계·null 시각화, eda_sohee.ipynb 셀 9–14 (bravedolphinhere)의 빈 문자열 집계와
> 결측 이유 표를 합쳤습니다.

- 원본 JSON을 다시 읽어 `data/README.md` 스키마의 모든 키를 레코드마다 검사합니다. 로더(`ann_df`)는
  키 없음과 null을 모두 NaN으로 만들기 때문입니다. 함수는 `scan_missing_values`입니다.
- 결측 유형: `키 없음`, `null`, `빈 문자열`(공백만 있는 값 포함), `빈 배열/객체`, `NaN/무한대`,
  `bbox 내부 결측`. 0과 False는 결측으로 세지 않습니다(크기 0은 아래에서 따로 봅니다).
- `필수여부`는 스키마 기준입니다(M 필수, O 선택, C 조건부). O·C 키는 비어 있어도 정상일 수 있습니다.

In [ ]:
missing_scan_df = scan_missing_values(TRAIN_ANN_DIR, schema_df)
kinds = ["키 없음", "null", "빈 문자열", "빈 배열/객체", "NaN/무한대", "bbox 내부 결측"]
missing_table = pd.crosstab(missing_scan_df["key"], missing_scan_df["kind"]).reindex(
    columns=kinds, fill_value=0
)
n_checked = missing_scan_df.groupby("key").size()
missing_table.insert(0, "검사 수", n_checked)
missing_table["결측 비율(%)"] = (100 * missing_table[kinds].sum(axis=1) / n_checked).round(1)
missing_table = missing_table.join(schema_df.set_index("key")[["required", "description"]])
missing_table = missing_table[missing_table["결측 비율(%)"] > 0].sort_values(
    ["결측 비율(%)", "null"], ascending=False
)

# 학습에 바로 쓰는 키. 여기가 비면 라벨을 연결하거나 bbox를 만들 수 없다.
core_keys = [
    "images.id",
    "images.file_name",
    "images.width",
    "images.height",
    "annotations.id",
    "annotations.image_id",
    "annotations.category_id",
    "annotations.bbox",
    "categories.id",
]
core_missing = missing_scan_df[
    missing_scan_df["key"].isin(core_keys) & (missing_scan_df["kind"] != "값 있음")
]
print(
    f"[원본] 스키마 키 {n_checked.size}개 중 결측이 있는 키: {len(missing_table)}개 "
    f"(필수 M: {(missing_table['required'] == 'M').sum()}개)"
)
print(f"[원본] 학습에 쓰는 핵심 키의 결측: {len(core_missing)}건")
with pd.option_context("display.max_rows", 100):
    display(missing_table.rename(columns={"required": "필수여부", "description": "스키마 설명"}))

In [ ]:
# null이 있는 images 레코드가 어떤 약인지 본다.
# 한 약의 레코드가 모두 null이면 약 정보 자체가 빈 것이다.
null_scan = missing_scan_df[
    (missing_scan_df["section"] == "images") & (missing_scan_df["kind"] == "null")
]
null_records = (
    null_scan.groupby(["source_json", "record_index", "file_name", "drug_N"])["key"]
    .agg(n_null="size", null_keys=lambda s: ", ".join(k.removeprefix("images.") for k in s))
    .reset_index()
)
print(
    f"[원본] null 키가 있는 images 레코드: {len(null_records)}개, "
    f"사진 {null_records['file_name'].nunique()}장"
)
null_by_drug = null_records.groupby("drug_N").agg(
    null_records=("source_json", "size"), n_null_keys=("n_null", "max")
)
drug_info = ann_df.groupby("drug_N").agg(dl_name=("dl_name", "first"), all_records=("x", "size"))
null_by_drug = null_by_drug.join(drug_info)
print("[원본] 약별 null 레코드 수 / 그 약의 전체 레코드 수")
display(null_by_drug)
print("[원본] null인 키 목록 (중복 제거)")
display(null_records["null_keys"].value_counts().rename("레코드 수").to_frame())

In [ ]:
# 0은 결측으로 세지 않았지만, 크기(mm)의 0.0은 값이 없다는 뜻으로 쓰였다(kwon 정의서).
size_cols = ["thick", "leng_long", "leng_short"]
size_zero = ann_df[size_cols].eq(0)
print("[원본] 크기(mm)가 0인 bbox 수")
display(size_zero.sum().rename("0인 bbox 수").to_frame().T)
display(
    ann_df.loc[size_zero.any(axis=1)]
    .groupby(["drug_N", "dl_name", *size_cols])
    .size()
    .rename("bbox 수")
    .reset_index()
)

In [ ]:
# null 레코드의 알약(빨강)에도 bbox와 클래스가 연결되어 있는지 본다. 조합이 다른 사진 최대 4장.
null_examples = (
    null_records.assign(combo=null_records["file_name"].str.split("_").str[0])
    .drop_duplicates("combo")
    .head(4)
)
if null_examples.empty:
    print("[원본] null 레코드가 없습니다.")
else:
    fig, axes = image_grid(len(null_examples), cols=4)
    for ax, row in zip(axes, null_examples.itertuples(), strict=True):
        boxes = ann_df[ann_df["file_name"] == row.file_name]
        is_null = boxes["source_json"] == row.source_json
        title = f"[원본] {row.combo}\n빨강: {row.drug_N} (null 키 {row.n_null}개)"
        show_image(ax, TRAIN_IMG_DIR / row.file_name, title)
        draw_bboxes(ax, boxes[~is_null], color="gray")
        draw_bboxes(ax, boxes[is_null], color="red")
    plt.tight_layout()
    plt.show()

**결측을 어떻게 읽을까**

빈 값만 보고 "마크 없음", "분할선 없음"으로 확정하지 않습니다. 정보가 원래 없는 것인지 수집되지 않은
것인지는 데이터 설명과 실제 이미지를 함께 봐야 구분할 수 있습니다(eda_sohee 셀 10). 아래는 비어 있는
이유로 생각해 볼 수 있는 것입니다.

| 키 | 비어 있는 이유로 생각해 볼 수 있는 것 |
|---|---|
| `print_front`, `print_back` | 그 면에 식별문자가 없을 수 있음 |
| `line_front`, `line_back` | 분할선이 없을 수 있음 |
| `color_class2` | 두 번째 색이 없는 단색 알약일 수 있음 |
| `mark_code_*` | 마크가 없거나 마크 정보가 수집되지 않았을 수 있음 |
| `di_company_mf`, `di_company_mf_en` | 위탁제조사가 없거나 영문명이 제공되지 않았을 수 있음 |
| `annotations.segmentation` | 이 데이터는 bbox만 라벨링함 (스키마상 선택 O) |

- 빈 값 대부분은 사진이 아니라 JSON의 약 설명이 비어 있는 것입니다. `dl_company_en`처럼 사진으로 확인할
  수 없는 키도 있고, `print_back`처럼 사진과 비교해 볼 수 있는 키도 있습니다(eda_sohee 셀 13).
- `null` 18건은 모두 `K-016548`(가바토파정 100mg)의 레코드이고, 이 약의 레코드 18개 전부입니다.
  키는 있고 값이 `null`이며(키 없음 0건), 크기(mm)도 0입니다. 사진이 아니라 약 정보 하나가 통째로
  비어 있는 것입니다(원본 상태 관측값).
- 위 그림처럼 이 레코드에도 bbox와 클래스는 연결되어 있으므로 메타데이터 결측 자체는 bbox 학습에
  영향이 없습니다(JJAM 셀 37). 다만 `K-001900-016548-019607-033009` 70° 사진에서는 16548 박스가
  다른 약(33009)과 같은 좌표에 붙어 있고, 실제 16548 알약에는 박스가 없습니다. 이것은 결측이 아니라
  좌표 오류로, 5.2에서 봅니다(JJAM 셀 38).
- 메타데이터를 학습 입력으로 쓰지 않는 한 결측을 채울 필요는 없어 보입니다(dongwook 셀 8). 알약 실제
  크기를 쓰는 7.3에서는 크기 0인 약을 빼고 계산합니다.

## 5. 라벨 오류와 이상치

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 5.1 이미지 밖 좌표 — JJAM 22, 28, 31
- 5.2 다른 약과 좌표 중복 — kwon 58–59, jang 9의 정답 약 판정
- 5.3 빈 배경 박스 — kwon 70–73
- 5.4 area ≠ w×h — JJAM 46
- 5.5 면적·가로세로 비율 이상치 후보 (IQR, 오류 아님) — JJAM 22, eda_sohee 19–22
- 5.6 붙은·부서진 알약 — kwon 54–66
- 5.7 bbox 여백 (GrabCut 추정) [RUN_HEAVY] — JJAM 54–66
- 5.8 라벨 오류 요약표 — 문제 유형, 건수, 처리 방식, 보정 파일

## 6. 분포

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 6.1 클래스별 bbox 수와 불균형 단계 — kwon 35, 37, jang 15
- 6.2 bbox 크기 분포, 클래스별 크기 — jang 13, 17

## 7. 촬영 조건

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 7.1 조합별 각도 구성 — kwon 41, 43
- 7.2 각도·배경·조명별 bbox 크기와 비율 — dongwook 11, jang 19–20
- 7.3 알약 실제 크기와 픽셀 크기 — dongwook 13–17 (JJAM 과거 분석은 Git 이력에서 확인 후 보완)

## 8. 데이터 누수와 분할

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 8.1 희귀 약은 사실상 장면 하나 — kwon 46–53
- 8.2 같은 조합 유사도 (pHash) — kwon 90–107, sohee_eda 15의 aHash와 비교
- 8.3 다른 조합의 세팅 공유와 묶음 — kwon 109–118
- 8.4 분할 후보 비교 — kwon 120, sohee_eda 17, jang 20

## 9. 결론과 다음 작업

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 확정된 사실
- 결정할 것
- 학습 파이프라인으로 넘길 것